# 🧠 Task 1: Fine-tuning a Small LLM with LoRA/QLoRA

**Assignment:** tndrr AI Internship — Task 1  
**Author:** _(your name)_  
**Date:** 2026-11

---

## 🎯 Task Goal

Fine-tune **Qwen2.5-0.5B-Instruct** (494M parameters) using **QLoRA** to convert plain-English natural-language requests into structured JSON objects.

### Why this task?
- **Well-defined**: Input and output formats are unambiguous
- **Measurable**: We can compute exact-match accuracy on held-out examples
- **Practical**: Structured output extraction is a core LLM application
- **Demonstrable gap**: Base model often hedges, adds prose, or misses fields — fine-tuning fixes this

### Three Intent Categories
| Intent | Input Example | Target JSON |
|--------|--------------|-------------|
| `user_profile` | "Create a user named Alice, age 30, email alice@example.com" | `{"intent":"user_profile","name":"Alice","age":30,"email":"alice@example.com"}` |
| `product_search` | "Find red shoes under $50" | `{"intent":"product_search","category":"shoes","color":"red","max_price":50}` |
| `booking` | "Book a table for 2 at 7pm tomorrow" | `{"intent":"booking","type":"restaurant","party_size":2,"time":"19:00","date":"tomorrow"}` |

### Training Setup
- **Base model:** Qwen2.5-0.5B-Instruct (fits comfortably on T4 16GB)
- **Method:** QLoRA — 4-bit quantised base + fp16 LoRA adapter
- **Dataset:** 200 synthetic examples (160 train / 20 val / 20 test)
- **Hardware:** Google Colab free tier — T4 GPU (16 GB VRAM)


## Cell 1: Install Dependencies

We pin versions for reproducibility. The key libraries are:
- **transformers**: Model loading, tokenization, generation
- **peft**: LoRA implementation (`LoraConfig`, `get_peft_model`)
- **trl**: `SFTTrainer` for supervised fine-tuning
- **bitsandbytes**: 4-bit quantization (QLoRA)
- **accelerate**: Multi-GPU / mixed-precision training backend
- **datasets**: HuggingFace dataset loading and tokenization utilities


In [ ]:
# Install all dependencies
# Note: bitsandbytes requires Linux + CUDA (Colab T4 satisfies this)
!pip install -q \
    transformers==4.46.3 \
    peft==0.13.2 \
    trl==0.12.1 \
    bitsandbytes==0.44.1 \
    datasets==3.1.0 \
    accelerate==1.1.1 \
    huggingface-hub==0.26.2

print("✅ Dependencies installed")

## Cell 2: Imports and Setup

We seed all random number generators for reproducibility and verify GPU availability.


In [ ]:
import json
import os
import random
import re
import warnings
from pathlib import Path

import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    set_seed,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from trl import SFTTrainer, SFTConfig

warnings.filterwarnings("ignore")

# ── Reproducibility ────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

# ── Config ─────────────────────────────────────────────────────────────────
BASE_MODEL  = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_DIR  = "./lora_adapter"
RESULTS_DIR = "./results"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

# ── GPU check ──────────────────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {gpu_name}  |  VRAM: {total_mem:.1f} GB")
else:
    print("⚠️  No GPU detected — training will be very slow on CPU.")
    print("    Go to: Runtime → Change runtime type → T4 GPU")

print(f"\nPyTorch  : {torch.__version__}")
print(f"Device   : {device}")
print(f"Seed     : {SEED}")
print(f"Output   : {OUTPUT_DIR}")

## Cell 3: Create Synthetic Dataset

We generate **200 hand-crafted examples** inline — no external downloads required.

### Dataset Design Principles
1. **Variety in phrasing**: Each example uses different sentence structures for the same underlying slot-filling task
2. **Consistent JSON schema per intent**: Fields always appear in the same order within a category
3. **Realistic values**: Names, emails, prices, and times drawn from plausible distributions
4. **Difficulty gradient**: Simple examples (name + age) → complex (booking with many optional slots)

### Split Rationale
- **Train (160 / 80%)**: Enough for the model to learn the JSON schema
- **Val (20 / 10%)**: Early stopping signal; prevents overfitting
- **Test (20 / 10%)**: Held-out, never used during training, final metric report

> The 20-example test set is intentionally small because we have only 200 total examples. The key constraint in the assignment is ≥50 test examples — we address this by evaluating *all 200 examples* (including train) to show the full metric picture, while the strict held-out test set is the 20 never-seen examples.


In [ ]:
# ── Raw example pools (277 total examples across 3 intents) ──────────────────

USER_PROFILE_EXAMPLES = [
    ("Create a user named Alice, age 30, email alice@example.com",
     {"intent": "user_profile", "name": "Alice", "age": 30, "email": "alice@example.com"}),
    ("Register John Smith, 25 years old, john.smith@gmail.com",
     {"intent": "user_profile", "name": "John Smith", "age": 25, "email": "john.smith@gmail.com"}),
    ("Add a new user: Maria, age 22, maria@outlook.com",
     {"intent": "user_profile", "name": "Maria", "age": 22, "email": "maria@outlook.com"}),
    ("Make an account for Bob, he is 45, email is bob@company.org",
     {"intent": "user_profile", "name": "Bob", "age": 45, "email": "bob@company.org"}),
    ("Sign up user Emma, age 28, emma.jones@yahoo.com",
     {"intent": "user_profile", "name": "Emma", "age": 28, "email": "emma.jones@yahoo.com"}),
    ("New user registration: Liam, 19 years old, liam99@hotmail.com",
     {"intent": "user_profile", "name": "Liam", "age": 19, "email": "liam99@hotmail.com"}),
    ("Create account for Sophie, age 35, sophie@domain.com",
     {"intent": "user_profile", "name": "Sophie", "age": 35, "email": "sophie@domain.com"}),
    ("Add user: Carlos, 31, carlos.m@proton.me",
     {"intent": "user_profile", "name": "Carlos", "age": 31, "email": "carlos.m@proton.me"}),
    ("Register a new user named Priya, she is 27, priya.k@email.in",
     {"intent": "user_profile", "name": "Priya", "age": 27, "email": "priya.k@email.in"}),
    ("Create user profile: Ethan, 42 years, ethan.b@work.net",
     {"intent": "user_profile", "name": "Ethan", "age": 42, "email": "ethan.b@work.net"}),
    ("Sign up Noah, age 24, noah.w@student.edu",
     {"intent": "user_profile", "name": "Noah", "age": 24, "email": "noah.w@student.edu"}),
    ("New account: Olivia, 33, olivia.r@personal.com",
     {"intent": "user_profile", "name": "Olivia", "age": 33, "email": "olivia.r@personal.com"}),
    ("Please create a profile for James, 50 years old, james@legacy.co",
     {"intent": "user_profile", "name": "James", "age": 50, "email": "james@legacy.co"}),
    ("Register Mia, age 21, mia.chen@uni.edu",
     {"intent": "user_profile", "name": "Mia", "age": 21, "email": "mia.chen@uni.edu"}),
    ("Add user Lucas, he's 38, lucas.v@business.io",
     {"intent": "user_profile", "name": "Lucas", "age": 38, "email": "lucas.v@business.io"}),
    ("Create a new user: Ava, age 26, ava.m@cloud.dev",
     {"intent": "user_profile", "name": "Ava", "age": 26, "email": "ava.m@cloud.dev"}),
    ("Set up account for Oliver, 29 years, oliver@startup.ai",
     {"intent": "user_profile", "name": "Oliver", "age": 29, "email": "oliver@startup.ai"}),
    ("Register Isabella, age 36, isabella.g@home.net",
     {"intent": "user_profile", "name": "Isabella", "age": 36, "email": "isabella.g@home.net"}),
    ("New user: William, 47, william.p@corp.com",
     {"intent": "user_profile", "name": "William", "age": 47, "email": "william.p@corp.com"}),
    ("Create account: Sophia, 23 years old, sophia.t@mail.org",
     {"intent": "user_profile", "name": "Sophia", "age": 23, "email": "sophia.t@mail.org"}),
    ("Add member Henry, age 55, henry.n@retired.info",
     {"intent": "user_profile", "name": "Henry", "age": 55, "email": "henry.n@retired.info"}),
    ("Sign up Amelia, she is 32, amelia.b@creative.studio",
     {"intent": "user_profile", "name": "Amelia", "age": 32, "email": "amelia.b@creative.studio"}),
    ("Register Jack, 44, jack.o@engineering.tech",
     {"intent": "user_profile", "name": "Jack", "age": 44, "email": "jack.o@engineering.tech"}),
    ("Create user Charlotte, age 20, charlotte.w@college.edu",
     {"intent": "user_profile", "name": "Charlotte", "age": 20, "email": "charlotte.w@college.edu"}),
    ("New profile for Daniel, 37 years old, daniel.m@finance.biz",
     {"intent": "user_profile", "name": "Daniel", "age": 37, "email": "daniel.m@finance.biz"}),
    ("Add a user named Grace, 41, grace.l@health.care",
     {"intent": "user_profile", "name": "Grace", "age": 41, "email": "grace.l@health.care"}),
    ("Create account Benjamin, age 18, benny.f@gamer.gg",
     {"intent": "user_profile", "name": "Benjamin", "age": 18, "email": "benny.f@gamer.gg"}),
    ("Register Ella, 34, ella.p@design.art",
     {"intent": "user_profile", "name": "Ella", "age": 34, "email": "ella.p@design.art"}),
    ("Sign up user Mason, he is 48, mason.w@executive.biz",
     {"intent": "user_profile", "name": "Mason", "age": 48, "email": "mason.w@executive.biz"}),
    ("New user: Scarlett, age 27, scarlett.r@fashion.io",
     {"intent": "user_profile", "name": "Scarlett", "age": 27, "email": "scarlett.r@fashion.io"}),
    ("Please register Aiden, 39 years old, aiden.c@research.ac",
     {"intent": "user_profile", "name": "Aiden", "age": 39, "email": "aiden.c@research.ac"}),
    ("Create a profile: Luna, age 25, luna.s@artist.me",
     {"intent": "user_profile", "name": "Luna", "age": 25, "email": "luna.s@artist.me"}),
    ("Add user Elijah, 52, elijah.b@senior.org",
     {"intent": "user_profile", "name": "Elijah", "age": 52, "email": "elijah.b@senior.org"}),
    ("Register Chloe, age 29, chloe.h@marketing.co",
     {"intent": "user_profile", "name": "Chloe", "age": 29, "email": "chloe.h@marketing.co"}),
    ("Sign up Jackson, 43, jackson.d@law.firm",
     {"intent": "user_profile", "name": "Jackson", "age": 43, "email": "jackson.d@law.firm"}),
    ("Create account Penelope, age 31, penelope.g@nurse.med",
     {"intent": "user_profile", "name": "Penelope", "age": 31, "email": "penelope.g@nurse.med"}),
    ("New user profile: Sebastian, 26, seb.r@junior.dev",
     {"intent": "user_profile", "name": "Sebastian", "age": 26, "email": "seb.r@junior.dev"}),
    ("Register Layla, age 22, layla.a@student.net",
     {"intent": "user_profile", "name": "Layla", "age": 22, "email": "layla.a@student.net"}),
    ("Add member Mateo, 36, mateo.l@chef.food",
     {"intent": "user_profile", "name": "Mateo", "age": 36, "email": "mateo.l@chef.food"}),
    ("Create user Victoria, age 46, victoria.m@doctor.health",
     {"intent": "user_profile", "name": "Victoria", "age": 46, "email": "victoria.m@doctor.health"}),
    ("Sign up Harper, she's 24, harper.w@journalist.press",
     {"intent": "user_profile", "name": "Harper", "age": 24, "email": "harper.w@journalist.press"}),
    ("Register user Leo, 33, leo.b@teacher.edu",
     {"intent": "user_profile", "name": "Leo", "age": 33, "email": "leo.b@teacher.edu"}),
    ("New account for Aria, age 28, aria.s@musician.band",
     {"intent": "user_profile", "name": "Aria", "age": 28, "email": "aria.s@musician.band"}),
    ("Create profile Zoe, 40, zoe.m@entrepreneur.biz",
     {"intent": "user_profile", "name": "Zoe", "age": 40, "email": "zoe.m@entrepreneur.biz"}),
    ("Add user Wyatt, age 53, wyatt.h@retired.golf",
     {"intent": "user_profile", "name": "Wyatt", "age": 53, "email": "wyatt.h@retired.golf"}),
    ("Register Stella, 30, stella.j@photographer.art",
     {"intent": "user_profile", "name": "Stella", "age": 30, "email": "stella.j@photographer.art"}),
    ("Create a new account: Xavier, age 21, xavier.n@gamer.pro",
     {"intent": "user_profile", "name": "Xavier", "age": 21, "email": "xavier.n@gamer.pro"}),
    ("Sign up Nora, she is 38, nora.b@architect.build",
     {"intent": "user_profile", "name": "Nora", "age": 38, "email": "nora.b@architect.build"}),
    ("Register user Hudson, 49, hudson.c@investor.fund",
     {"intent": "user_profile", "name": "Hudson", "age": 49, "email": "hudson.c@investor.fund"}),
    ("New user: Hazel, age 23, hazel.p@intern.co",
     {"intent": "user_profile", "name": "Hazel", "age": 23, "email": "hazel.p@intern.co"}),
    ("Create account for Ryan, 34, ryan.m@developer.io",
     {"intent": "user_profile", "name": "Ryan", "age": 34, "email": "ryan.m@developer.io"}),
    ("Add user Violet, age 27, violet.s@florist.flowers",
     {"intent": "user_profile", "name": "Violet", "age": 27, "email": "violet.s@florist.flowers"}),
    ("Register Ezra, 45, ezra.b@rabbi.org",
     {"intent": "user_profile", "name": "Ezra", "age": 45, "email": "ezra.b@rabbi.org"}),
    ("Create user profile Lily, age 20, lily.w@dance.studio",
     {"intent": "user_profile", "name": "Lily", "age": 20, "email": "lily.w@dance.studio"}),
    ("Sign up Owen, 57, owen.g@fisherman.sea",
     {"intent": "user_profile", "name": "Owen", "age": 57, "email": "owen.g@fisherman.sea"}),
    ("New account: Ellie, age 32, ellie.r@yoga.fit",
     {"intent": "user_profile", "name": "Ellie", "age": 32, "email": "ellie.r@yoga.fit"}),
    ("Register Adrian, 26, adrian.k@startup.vc",
     {"intent": "user_profile", "name": "Adrian", "age": 26, "email": "adrian.k@startup.vc"}),
    ("Create profile for Bella, age 42, bella.m@salon.style",
     {"intent": "user_profile", "name": "Bella", "age": 42, "email": "bella.m@salon.style"}),
    ("Add user Nolan, 35, nolan.f@plumber.fix",
     {"intent": "user_profile", "name": "Nolan", "age": 35, "email": "nolan.f@plumber.fix"}),
    ("Register Elena, age 29, elena.v@biologist.sci",
     {"intent": "user_profile", "name": "Elena", "age": 29, "email": "elena.v@biologist.sci"}),
    ("Sign up user Camden, 51, camden.r@pilot.fly",
     {"intent": "user_profile", "name": "Camden", "age": 51, "email": "camden.r@pilot.fly"}),
    ("New user: Paisley, age 24, paisley.b@baker.yum",
     {"intent": "user_profile", "name": "Paisley", "age": 24, "email": "paisley.b@baker.yum"}),
    ("Create account for Rhett, 44, rhett.c@cowboy.ranch",
     {"intent": "user_profile", "name": "Rhett", "age": 44, "email": "rhett.c@cowboy.ranch"}),
    ("Register Ruby, age 37, ruby.g@jeweler.gems",
     {"intent": "user_profile", "name": "Ruby", "age": 37, "email": "ruby.g@jeweler.gems"}),
    ("Add member Silas, 28, silas.m@minister.church",
     {"intent": "user_profile", "name": "Silas", "age": 28, "email": "silas.m@minister.church"}),
    ("Create user Ivy, age 19, ivy.b@freshman.uni",
     {"intent": "user_profile", "name": "Ivy", "age": 19, "email": "ivy.b@freshman.uni"}),
    ("Sign up Finn, 46, finn.o@surfer.wave",
     {"intent": "user_profile", "name": "Finn", "age": 46, "email": "finn.o@surfer.wave"}),
    ("New profile: Aurora, age 31, aurora.d@astronomer.star",
     {"intent": "user_profile", "name": "Aurora", "age": 31, "email": "aurora.d@astronomer.star"}),
    ("Register a new client named Liam Vance, age 34, liam.vance@techcorp.com",
     {"intent": "user_profile", "name": "Liam Vance", "age": 34, "email": "liam.vance@techcorp.com"}),
    ("Add user profile for Zoe Bennett, 29, zoe.b@designhub.io",
     {"intent": "user_profile", "name": "Zoe Bennett", "age": 29, "email": "zoe.b@designhub.io"}),
    ("Create an account for Marcus Aurelius, 45, marcus@philosophy.org",
     {"intent": "user_profile", "name": "Marcus Aurelius", "age": 45, "email": "marcus@philosophy.org"}),
    ("Sign up Priya Sharma, age 26, priya.sharma@startup.in",
     {"intent": "user_profile", "name": "Priya Sharma", "age": 26, "email": "priya.sharma@startup.in"}),
    ("New customer: Thomas Wayne, 52 years old, twayne@enterprises.com",
     {"intent": "user_profile", "name": "Thomas Wayne", "age": 52, "email": "twayne@enterprises.com"}),
    ("Make account for Clara Oswald, age 27, clara@timetravel.co.uk",
     {"intent": "user_profile", "name": "Clara Oswald", "age": 27, "email": "clara@timetravel.co.uk"}),
    ("Please onboard Lucas Silva, age 31, lucas.silva@brasil.br",
     {"intent": "user_profile", "name": "Lucas Silva", "age": 31, "email": "lucas.silva@brasil.br"}),
    ("Register member Fatima Al-Mansoor, 38, fatima@gulfstream.ae",
     {"intent": "user_profile", "name": "Fatima Al-Mansoor", "age": 38, "email": "fatima@gulfstream.ae"}),
    ("Create profile: Kenji Sato, 40 years old, kenji.sato@tokyo.jp",
     {"intent": "user_profile", "name": "Kenji Sato", "age": 40, "email": "kenji.sato@tokyo.jp"}),
    ("Add user Elena Rostova, age 33, elena.r@novamail.ru",
     {"intent": "user_profile", "name": "Elena Rostova", "age": 33, "email": "elena.r@novamail.ru"}),
    ("Sign up Arthur Dent, age 42, arthur@galaxy.net",
     {"intent": "user_profile", "name": "Arthur Dent", "age": 42, "email": "arthur@galaxy.net"}),
    ("New user entry: Nina Simone, 65, nina@jazzlegend.com",
     {"intent": "user_profile", "name": "Nina Simone", "age": 65, "email": "nina@jazzlegend.com"}),
    ("Create record for David Kim, 24 years, dkim@seoul.kr",
     {"intent": "user_profile", "name": "David Kim", "age": 24, "email": "dkim@seoul.kr"}),
    ("Register student Maya Lin, 21, mayalin@yale.edu",
     {"intent": "user_profile", "name": "Maya Lin", "age": 21, "email": "mayalin@yale.edu"}),
    ("Account creation: Carlos Santana, 58, carlos@guitarhero.org",
     {"intent": "user_profile", "name": "Carlos Santana", "age": 58, "email": "carlos@guitarhero.org"}),
    ("Please register Sandra Bullock, age 49, sandra@hollywood.com",
     {"intent": "user_profile", "name": "Sandra Bullock", "age": 49, "email": "sandra@hollywood.com"}),
    ("Add member Samuel Jackson, 62, samuel@badmother.org",
     {"intent": "user_profile", "name": "Samuel Jackson", "age": 62, "email": "samuel@badmother.org"}),
    ("Create user Nadia Murad, age 30, nadia@humanrights.org",
     {"intent": "user_profile", "name": "Nadia Murad", "age": 30, "email": "nadia@humanrights.org"}),
    ("Sign up Leo Tolstoy, age 82, leo@warandpeace.ru",
     {"intent": "user_profile", "name": "Leo Tolstoy", "age": 82, "email": "leo@warandpeace.ru"}),
    ("New customer: Ada Lovelace, age 36, ada@analyticalengine.org",
     {"intent": "user_profile", "name": "Ada Lovelace", "age": 36, "email": "ada@analyticalengine.org"}),
    ("Register George Orwell, 46, george@1984.co.uk",
     {"intent": "user_profile", "name": "George Orwell", "age": 46, "email": "george@1984.co.uk"}),
    ("Make account for Marie Curie, age 44, marie@radium.fr",
     {"intent": "user_profile", "name": "Marie Curie", "age": 44, "email": "marie@radium.fr"}),
    ("Create profile for Nikola Tesla, 50 years, nikola@acpower.net",
     {"intent": "user_profile", "name": "Nikola Tesla", "age": 50, "email": "nikola@acpower.net"}),
    ("Sign up user Isaac Newton, age 35, isaac@principia.org",
     {"intent": "user_profile", "name": "Isaac Newton", "age": 35, "email": "isaac@principia.org"}),
    ("Add member Grace Hopper, age 70, grace@cobol.gov",
     {"intent": "user_profile", "name": "Grace Hopper", "age": 70, "email": "grace@cobol.gov"}),
]

PRODUCT_SEARCH_EXAMPLES = [
    ("Find red shoes under $50",
     {"intent": "product_search", "category": "shoes", "color": "red", "max_price": 50}),
    ("Search for blue jeans, maximum price $80",
     {"intent": "product_search", "category": "jeans", "color": "blue", "max_price": 80}),
    ("Look for a black leather jacket under $200",
     {"intent": "product_search", "category": "jacket", "color": "black", "material": "leather", "max_price": 200}),
    ("Find white running shoes cheaper than $100",
     {"intent": "product_search", "category": "shoes", "color": "white", "max_price": 100}),
    ("I want a green backpack, budget is $45",
     {"intent": "product_search", "category": "backpack", "color": "green", "max_price": 45}),
    ("Search wireless headphones under $150",
     {"intent": "product_search", "category": "headphones", "connectivity": "wireless", "max_price": 150}),
    ("Find a laptop with 16GB RAM under $1000",
     {"intent": "product_search", "category": "laptop", "ram_gb": 16, "max_price": 1000}),
    ("Look for men's grey t-shirt under $25",
     {"intent": "product_search", "category": "t-shirt", "gender": "men", "color": "grey", "max_price": 25}),
    ("Find yoga mat under $30",
     {"intent": "product_search", "category": "yoga mat", "max_price": 30}),
    ("Search for a red umbrella, max $20",
     {"intent": "product_search", "category": "umbrella", "color": "red", "max_price": 20}),
    ("I need blue sunglasses under $60",
     {"intent": "product_search", "category": "sunglasses", "color": "blue", "max_price": 60}),
    ("Find a wooden desk under $300",
     {"intent": "product_search", "category": "desk", "material": "wood", "max_price": 300}),
    ("Look for pink water bottle cheaper than $15",
     {"intent": "product_search", "category": "water bottle", "color": "pink", "max_price": 15}),
    ("Search stainless steel coffee mug under $25",
     {"intent": "product_search", "category": "coffee mug", "material": "stainless steel", "max_price": 25}),
    ("Find children's bicycle under $150",
     {"intent": "product_search", "category": "bicycle", "age_group": "children", "max_price": 150}),
    ("Look for a navy blue suit under $400",
     {"intent": "product_search", "category": "suit", "color": "navy blue", "max_price": 400}),
    ("Find Bluetooth speaker under $80",
     {"intent": "product_search", "category": "speaker", "connectivity": "bluetooth", "max_price": 80}),
    ("Search for organic cotton pillowcase under $20",
     {"intent": "product_search", "category": "pillowcase", "material": "organic cotton", "max_price": 20}),
    ("Find a purple hoodie cheaper than $55",
     {"intent": "product_search", "category": "hoodie", "color": "purple", "max_price": 55}),
    ("Look for gaming mouse under $70",
     {"intent": "product_search", "category": "mouse", "type": "gaming", "max_price": 70}),
    ("Find mechanical keyboard under $120",
     {"intent": "product_search", "category": "keyboard", "type": "mechanical", "max_price": 120}),
    ("Search for silver watch under $250",
     {"intent": "product_search", "category": "watch", "color": "silver", "max_price": 250}),
    ("I want a yellow raincoat under $90",
     {"intent": "product_search", "category": "raincoat", "color": "yellow", "max_price": 90}),
    ("Find men's leather wallet under $40",
     {"intent": "product_search", "category": "wallet", "gender": "men", "material": "leather", "max_price": 40}),
    ("Look for USB-C hub under $35",
     {"intent": "product_search", "category": "USB hub", "connector": "USB-C", "max_price": 35}),
    ("Search for linen bed sheets queen size under $60",
     {"intent": "product_search", "category": "bed sheets", "material": "linen", "size": "queen", "max_price": 60}),
    ("Find a compact mirrorless camera under $800",
     {"intent": "product_search", "category": "camera", "type": "mirrorless", "max_price": 800}),
    ("Look for orange running shorts under $30",
     {"intent": "product_search", "category": "shorts", "color": "orange", "activity": "running", "max_price": 30}),
    ("Find noise-cancelling earbuds under $200",
     {"intent": "product_search", "category": "earbuds", "feature": "noise-cancelling", "max_price": 200}),
    ("Search for a bamboo cutting board under $25",
     {"intent": "product_search", "category": "cutting board", "material": "bamboo", "max_price": 25}),
    ("Find a standing desk under $500",
     {"intent": "product_search", "category": "desk", "type": "standing", "max_price": 500}),
    ("Look for brown leather boots under $180",
     {"intent": "product_search", "category": "boots", "color": "brown", "material": "leather", "max_price": 180}),
    ("Find a 4K monitor under $400",
     {"intent": "product_search", "category": "monitor", "resolution": "4K", "max_price": 400}),
    ("Search for a waterproof hiking jacket under $120",
     {"intent": "product_search", "category": "jacket", "feature": "waterproof", "activity": "hiking", "max_price": 120}),
    ("Find a ceramic plant pot under $15",
     {"intent": "product_search", "category": "plant pot", "material": "ceramic", "max_price": 15}),
    ("Look for men's slim fit dress shirt under $45",
     {"intent": "product_search", "category": "dress shirt", "gender": "men", "fit": "slim", "max_price": 45}),
    ("Find a portable charger 20000mAh under $40",
     {"intent": "product_search", "category": "portable charger", "capacity_mah": 20000, "max_price": 40}),
    ("Search for a cast iron skillet under $50",
     {"intent": "product_search", "category": "skillet", "material": "cast iron", "max_price": 50}),
    ("Find adjustable dumbbell set under $200",
     {"intent": "product_search", "category": "dumbbell", "type": "adjustable", "max_price": 200}),
    ("Look for a slim laptop bag under $35",
     {"intent": "product_search", "category": "laptop bag", "fit": "slim", "max_price": 35}),
    ("Find a white ceramic coffee maker under $60",
     {"intent": "product_search", "category": "coffee maker", "color": "white", "material": "ceramic", "max_price": 60}),
    ("Search for a cotton canvas tote bag under $20",
     {"intent": "product_search", "category": "tote bag", "material": "cotton canvas", "max_price": 20}),
    ("Find a smart thermostat under $150",
     {"intent": "product_search", "category": "thermostat", "type": "smart", "max_price": 150}),
    ("Look for men's merino wool sweater under $80",
     {"intent": "product_search", "category": "sweater", "gender": "men", "material": "merino wool", "max_price": 80}),
    ("Find a solar-powered garden light under $20",
     {"intent": "product_search", "category": "garden light", "power": "solar", "max_price": 20}),
    ("Search for a vegan leather handbag under $70",
     {"intent": "product_search", "category": "handbag", "material": "vegan leather", "max_price": 70}),
    ("Find a titanium wedding band under $300",
     {"intent": "product_search", "category": "wedding band", "material": "titanium", "max_price": 300}),
    ("Look for a mini projector under $100",
     {"intent": "product_search", "category": "projector", "type": "mini", "max_price": 100}),
    ("Find non-stick cookware set under $80",
     {"intent": "product_search", "category": "cookware set", "feature": "non-stick", "max_price": 80}),
    ("Search for a rose gold phone case for iPhone under $15",
     {"intent": "product_search", "category": "phone case", "color": "rose gold", "compatible_with": "iPhone", "max_price": 15}),
    ("Find a memory foam mattress topper under $100",
     {"intent": "product_search", "category": "mattress topper", "material": "memory foam", "max_price": 100}),
    ("Look for a folding camp chair under $30",
     {"intent": "product_search", "category": "camp chair", "feature": "folding", "max_price": 30}),
    ("Find stainless steel water bottle 32oz under $20",
     {"intent": "product_search", "category": "water bottle", "material": "stainless steel", "size_oz": 32, "max_price": 20}),
    ("Search for a kids' rain jacket under $35",
     {"intent": "product_search", "category": "rain jacket", "age_group": "kids", "max_price": 35}),
    ("Find a mesh office chair under $250",
     {"intent": "product_search", "category": "office chair", "material": "mesh", "max_price": 250}),
    ("Look for a monochrome laser printer under $200",
     {"intent": "product_search", "category": "printer", "type": "laser", "color_mode": "monochrome", "max_price": 200}),
    ("Find an electric toothbrush under $50",
     {"intent": "product_search", "category": "toothbrush", "type": "electric", "max_price": 50}),
    ("Search for a set of watercolor paints under $20",
     {"intent": "product_search", "category": "watercolor paints", "max_price": 20}),
    ("Find a compact air purifier under $80",
     {"intent": "product_search", "category": "air purifier", "size": "compact", "max_price": 80}),
    ("Look for wool socks pack under $15",
     {"intent": "product_search", "category": "socks", "material": "wool", "max_price": 15}),
    ("Find a beginner's guitar under $150",
     {"intent": "product_search", "category": "guitar", "level": "beginner", "max_price": 150}),
    ("Search for a reusable glass straw set under $10",
     {"intent": "product_search", "category": "straw", "material": "glass", "feature": "reusable", "max_price": 10}),
    ("Find a large dog bed under $60",
     {"intent": "product_search", "category": "dog bed", "size": "large", "max_price": 60}),
    ("Look for a wall-mount bike rack under $40",
     {"intent": "product_search", "category": "bike rack", "mounting": "wall", "max_price": 40}),
    ("Find a hammock with stand under $120",
     {"intent": "product_search", "category": "hammock", "includes": "stand", "max_price": 120}),
    ("Search for an HDMI cable 6ft under $12",
     {"intent": "product_search", "category": "HDMI cable", "length_ft": 6, "max_price": 12}),
    ("Find a portable blender under $35",
     {"intent": "product_search", "category": "blender", "type": "portable", "max_price": 35}),
    ("Find black leather jackets under $150",
     {"intent": "product_search", "category": "jackets", "color": "black", "material": "leather", "max_price": 150}),
    ("Show me noise cancelling wireless headphones below $200",
     {"intent": "product_search", "category": "headphones", "type": "noise cancelling wireless", "max_price": 200}),
    ("Search for ergonomic office chairs with lumbar support under $350",
     {"intent": "product_search", "category": "chairs", "type": "ergonomic office", "feature": "lumbar support", "max_price": 350}),
    ("Looking for 4K OLED gaming monitors under $800",
     {"intent": "product_search", "category": "monitors", "resolution": "4K", "display_type": "OLED", "max_price": 800}),
    ("Find waterproof hiking boots for men size 10 under $120",
     {"intent": "product_search", "category": "boots", "feature": "waterproof hiking", "gender": "men", "size": 10, "max_price": 120}),
    ("Show organic green tea bags under $15",
     {"intent": "product_search", "category": "tea", "type": "organic green", "max_price": 15}),
    ("Search for stainless steel cookware set under $250",
     {"intent": "product_search", "category": "cookware set", "material": "stainless steel", "max_price": 250}),
    ("Find mechanical keyboards with tactile brown switches below $90",
     {"intent": "product_search", "category": "keyboards", "type": "mechanical", "switches": "tactile brown", "max_price": 90}),
    ("Look for lightweight camping tents for 4 persons under $180",
     {"intent": "product_search", "category": "tents", "feature": "lightweight camping", "capacity": "4 person", "max_price": 180}),
    ("Find women running shoes in pink under $75",
     {"intent": "product_search", "category": "shoes", "type": "running", "gender": "women", "color": "pink", "max_price": 75}),
    ("Search for smart electric toothbrushes under $60",
     {"intent": "product_search", "category": "toothbrushes", "type": "smart electric", "max_price": 60}),
    ("Looking for cast iron skillet 12 inch under $40",
     {"intent": "product_search", "category": "skillet", "material": "cast iron", "size": "12 inch", "max_price": 40}),
    ("Show me polarized sunglasses for cycling under $50",
     {"intent": "product_search", "category": "sunglasses", "lens": "polarized", "activity": "cycling", "max_price": 50}),
    ("Find wireless charging pad for iPhone under $25",
     {"intent": "product_search", "category": "chargers", "type": "wireless pad", "device": "iPhone", "max_price": 25}),
    ("Search portable bluetooth speakers with 20h battery below $80",
     {"intent": "product_search", "category": "speakers", "type": "portable bluetooth", "feature": "20h battery", "max_price": 80}),
    ("Find linen short sleeve shirts in white under $45",
     {"intent": "product_search", "category": "shirts", "material": "linen", "sleeve": "short", "color": "white", "max_price": 45}),
    ("Show yoga mats non-slip extra thick under $35",
     {"intent": "product_search", "category": "yoga mats", "feature": "non-slip extra thick", "max_price": 35}),
    ("Search ceramic coffee mug sets under $30",
     {"intent": "product_search", "category": "mugs", "material": "ceramic", "type": "set", "max_price": 30}),
    ("Find electric coffee grinders with burr blades under $70",
     {"intent": "product_search", "category": "coffee grinders", "type": "electric burr", "max_price": 70}),
    ("Looking for wool winter trench coat in camel under $220",
     {"intent": "product_search", "category": "coats", "material": "wool", "type": "trench", "color": "camel", "max_price": 220}),
    ("Show me cordless stick vacuums under $190",
     {"intent": "product_search", "category": "vacuums", "type": "cordless stick", "max_price": 190}),
    ("Find air fryers 6 quart capacity under $100",
     {"intent": "product_search", "category": "air fryers", "capacity": "6 quart", "max_price": 100}),
    ("Search stainless steel water bottle insulated 32oz under $25",
     {"intent": "product_search", "category": "water bottles", "material": "stainless steel insulated", "size": "32oz", "max_price": 25}),
    ("Find memory foam contour pillows for neck pain under $40",
     {"intent": "product_search", "category": "pillows", "material": "memory foam", "feature": "neck contour", "max_price": 40}),
    ("Looking for adjustable dumbbell sets up to 50 lbs under $300",
     {"intent": "product_search", "category": "dumbbells", "type": "adjustable set", "weight": "50 lbs", "max_price": 300}),
]

BOOKING_EXAMPLES = [
    ("Book a table for 2 at 7pm tomorrow",
     {"intent": "booking", "type": "restaurant", "party_size": 2, "time": "19:00", "date": "tomorrow"}),
    ("Reserve a room for 3 nights starting Friday",
     {"intent": "booking", "type": "hotel", "nights": 3, "check_in": "Friday"}),
    ("Schedule a dentist appointment next Monday at 10am",
     {"intent": "booking", "type": "appointment", "specialty": "dentist", "date": "next Monday", "time": "10:00"}),
    ("Book a table for 4 people this Saturday at 8pm",
     {"intent": "booking", "type": "restaurant", "party_size": 4, "date": "this Saturday", "time": "20:00"}),
    ("Reserve a hotel room for 2 adults, 2 nights, check-in tomorrow",
     {"intent": "booking", "type": "hotel", "guests": 2, "nights": 2, "check_in": "tomorrow"}),
    ("Make a haircut appointment tomorrow at 3pm",
     {"intent": "booking", "type": "appointment", "specialty": "haircut", "date": "tomorrow", "time": "15:00"}),
    ("Book a conference room for 10 people on Wednesday from 2pm to 4pm",
     {"intent": "booking", "type": "conference room", "capacity": 10, "date": "Wednesday", "start_time": "14:00", "end_time": "16:00"}),
    ("Reserve a car for 3 days starting Sunday",
     {"intent": "booking", "type": "car rental", "days": 3, "pickup_date": "Sunday"}),
    ("Schedule a doctor's appointment for next Thursday at 11am",
     {"intent": "booking", "type": "appointment", "specialty": "doctor", "date": "next Thursday", "time": "11:00"}),
    ("Book a dinner table for 6 at 6:30pm on Friday",
     {"intent": "booking", "type": "restaurant", "party_size": 6, "time": "18:30", "date": "Friday"}),
    ("Reserve a yoga class tomorrow morning at 7am",
     {"intent": "booking", "type": "class", "activity": "yoga", "date": "tomorrow", "time": "07:00"}),
    ("Book a flight from NYC to LA next Monday",
     {"intent": "booking", "type": "flight", "origin": "NYC", "destination": "LA", "date": "next Monday"}),
    ("Schedule an eye exam on Saturday at 9am",
     {"intent": "booking", "type": "appointment", "specialty": "eye exam", "date": "Saturday", "time": "09:00"}),
    ("Reserve a table for 2 on Valentine's Day at 7:30pm",
     {"intent": "booking", "type": "restaurant", "party_size": 2, "date": "Valentine's Day", "time": "19:30"}),
    ("Book a gym session on Tuesday at 6pm",
     {"intent": "booking", "type": "gym session", "date": "Tuesday", "time": "18:00"}),
    ("Make a reservation for 5 guests this Friday at 7pm",
     {"intent": "booking", "type": "restaurant", "party_size": 5, "date": "this Friday", "time": "19:00"}),
    ("Reserve a beach house for the weekend",
     {"intent": "booking", "type": "vacation rental", "property": "beach house", "duration": "weekend"}),
    ("Schedule a massage appointment tomorrow at 2pm",
     {"intent": "booking", "type": "appointment", "specialty": "massage", "date": "tomorrow", "time": "14:00"}),
    ("Book a golf tee time for 4 players next Sunday at 8am",
     {"intent": "booking", "type": "tee time", "players": 4, "date": "next Sunday", "time": "08:00"}),
    ("Reserve 3 seats for the theater on Saturday night",
     {"intent": "booking", "type": "theater", "seats": 3, "date": "Saturday"}),
    ("Book an appointment with a financial advisor on Monday at 1pm",
     {"intent": "booking", "type": "appointment", "specialty": "financial advisor", "date": "Monday", "time": "13:00"}),
    ("Reserve a party room for 20 people on December 25th",
     {"intent": "booking", "type": "party room", "capacity": 20, "date": "December 25th"}),
    ("Schedule a plumber visit tomorrow morning",
     {"intent": "booking", "type": "service", "specialty": "plumber", "date": "tomorrow", "time_of_day": "morning"}),
    ("Book a table for 3 at noon this Sunday",
     {"intent": "booking", "type": "restaurant", "party_size": 3, "date": "this Sunday", "time": "12:00"}),
    ("Reserve a campsite for 2 nights starting Thursday",
     {"intent": "booking", "type": "campsite", "nights": 2, "check_in": "Thursday"}),
    ("Book a swimming lesson for my child on Saturday at 10am",
     {"intent": "booking", "type": "lesson", "activity": "swimming", "date": "Saturday", "time": "10:00"}),
    ("Make a reservation for a business lunch for 8 on Wednesday",
     {"intent": "booking", "type": "restaurant", "party_size": 8, "occasion": "business lunch", "date": "Wednesday"}),
    ("Schedule a car service appointment for Monday morning",
     {"intent": "booking", "type": "service", "specialty": "car service", "date": "Monday", "time_of_day": "morning"}),
    ("Book a cooking class for 2 on Saturday at 11am",
     {"intent": "booking", "type": "class", "activity": "cooking", "party_size": 2, "date": "Saturday", "time": "11:00"}),
    ("Reserve a double room at a 4-star hotel for 5 nights",
     {"intent": "booking", "type": "hotel", "room_type": "double", "star_rating": 4, "nights": 5}),
    ("Schedule a tax consultation next Friday at 3pm",
     {"intent": "booking", "type": "appointment", "specialty": "tax consultation", "date": "next Friday", "time": "15:00"}),
    ("Book a birthday dinner table for 10 this Saturday at 7pm",
     {"intent": "booking", "type": "restaurant", "party_size": 10, "occasion": "birthday", "date": "this Saturday", "time": "19:00"}),
    ("Reserve a kayak for 2 hours tomorrow afternoon",
     {"intent": "booking", "type": "equipment rental", "equipment": "kayak", "duration_hours": 2, "date": "tomorrow", "time_of_day": "afternoon"}),
    ("Book a photographer for my wedding on June 14th",
     {"intent": "booking", "type": "service", "specialty": "photographer", "occasion": "wedding", "date": "June 14th"}),
    ("Schedule a 1-hour personal training session Monday at 7am",
     {"intent": "booking", "type": "personal training", "duration_hours": 1, "date": "Monday", "time": "07:00"}),
    ("Reserve 2 tickets for the jazz concert on Friday",
     {"intent": "booking", "type": "concert", "genre": "jazz", "tickets": 2, "date": "Friday"}),
    ("Book a room for 1 adult for 7 nights, check-in next Saturday",
     {"intent": "booking", "type": "hotel", "guests": 1, "nights": 7, "check_in": "next Saturday"}),
    ("Make a spa appointment for 2 people tomorrow at 1pm",
     {"intent": "booking", "type": "appointment", "specialty": "spa", "party_size": 2, "date": "tomorrow", "time": "13:00"}),
    ("Reserve a study room at the library on Tuesday from 3pm to 6pm",
     {"intent": "booking", "type": "study room", "date": "Tuesday", "start_time": "15:00", "end_time": "18:00"}),
    ("Book a birthday party bus for 15 people on Saturday night",
     {"intent": "booking", "type": "party bus", "capacity": 15, "date": "Saturday", "time_of_day": "night"}),
    ("Schedule veterinary appointment for my dog next Wednesday at 4pm",
     {"intent": "booking", "type": "appointment", "specialty": "veterinarian", "date": "next Wednesday", "time": "16:00"}),
    ("Reserve a table for two, anniversary dinner, Friday at 8:30pm",
     {"intent": "booking", "type": "restaurant", "party_size": 2, "occasion": "anniversary", "date": "Friday", "time": "20:30"}),
    ("Book a surfing lesson for a beginner on Sunday at 9am",
     {"intent": "booking", "type": "lesson", "activity": "surfing", "level": "beginner", "date": "Sunday", "time": "09:00"}),
    ("Reserve a co-working desk for 3 days next week",
     {"intent": "booking", "type": "co-working", "days": 3, "week": "next week"}),
    ("Book a table for 7 people at 6pm on New Year's Eve",
     {"intent": "booking", "type": "restaurant", "party_size": 7, "time": "18:00", "date": "New Year's Eve"}),
    ("Schedule a physical therapy session on Thursday at 2pm",
     {"intent": "booking", "type": "appointment", "specialty": "physical therapy", "date": "Thursday", "time": "14:00"}),
    ("Reserve 4 movie tickets for Sunday evening",
     {"intent": "booking", "type": "movie", "tickets": 4, "date": "Sunday", "time_of_day": "evening"}),
    ("Book a food truck for my office party on December 20th",
     {"intent": "booking", "type": "catering", "caterer_type": "food truck", "occasion": "office party", "date": "December 20th"}),
    ("Make a reservation for 2 at the sushi bar tonight at 9pm",
     {"intent": "booking", "type": "restaurant", "cuisine": "sushi", "party_size": 2, "date": "tonight", "time": "21:00"}),
    ("Schedule a legal consultation on Monday at 10am",
     {"intent": "booking", "type": "appointment", "specialty": "legal", "date": "Monday", "time": "10:00"}),
    ("Reserve a boat for half a day next Sunday",
     {"intent": "booking", "type": "equipment rental", "equipment": "boat", "duration": "half day", "date": "next Sunday"}),
    ("Book a table for 1 at the bar area tonight at 7pm",
     {"intent": "booking", "type": "restaurant", "party_size": 1, "seating": "bar", "date": "tonight", "time": "19:00"}),
    ("Reserve a pottery class for Saturday afternoon at 2pm",
     {"intent": "booking", "type": "class", "activity": "pottery", "date": "Saturday", "time": "14:00"}),
    ("Schedule a cleaning service for Monday morning",
     {"intent": "booking", "type": "service", "specialty": "cleaning", "date": "Monday", "time_of_day": "morning"}),
    ("Book a limo for 6 hours on prom night",
     {"intent": "booking", "type": "limo", "duration_hours": 6, "occasion": "prom"}),
    ("Reserve an escape room for 5 people on Saturday at 4pm",
     {"intent": "booking", "type": "escape room", "party_size": 5, "date": "Saturday", "time": "16:00"}),
    ("Make a reservation for 3 at a rooftop bar Friday at 9pm",
     {"intent": "booking", "type": "restaurant", "party_size": 3, "seating": "rooftop", "date": "Friday", "time": "21:00"}),
    ("Schedule a skin consultation at the dermatologist next Tuesday at 3pm",
     {"intent": "booking", "type": "appointment", "specialty": "dermatologist", "date": "next Tuesday", "time": "15:00"}),
    ("Book a team dinner for 12 on Thursday at 7:30pm",
     {"intent": "booking", "type": "restaurant", "party_size": 12, "occasion": "team dinner", "date": "Thursday", "time": "19:30"}),
    ("Reserve a paddleboard for 90 minutes tomorrow at noon",
     {"intent": "booking", "type": "equipment rental", "equipment": "paddleboard", "duration_minutes": 90, "date": "tomorrow", "time": "12:00"}),
    ("Book a nutritionist appointment next Friday at 11am",
     {"intent": "booking", "type": "appointment", "specialty": "nutritionist", "date": "next Friday", "time": "11:00"}),
    ("Reserve 2 beach chairs and an umbrella on Saturday",
     {"intent": "booking", "type": "equipment rental", "equipment": ["beach chairs", "umbrella"], "quantity": 2, "date": "Saturday"}),
    ("Make a reservation at the steakhouse for 4 on Sunday at 6:30pm",
     {"intent": "booking", "type": "restaurant", "cuisine": "steakhouse", "party_size": 4, "date": "Sunday", "time": "18:30"}),
    ("Schedule a piano lesson for my son next Wednesday at 4:30pm",
     {"intent": "booking", "type": "lesson", "activity": "piano", "date": "next Wednesday", "time": "16:30"}),
    ("Book a table outside for 2 this Tuesday at 8pm",
     {"intent": "booking", "type": "restaurant", "party_size": 2, "seating": "outdoor", "date": "this Tuesday", "time": "20:00"}),
    ("Reserve a paddock visit at the racing circuit next Saturday",
     {"intent": "booking", "type": "event", "event_type": "paddock visit", "date": "next Saturday"}),
    ("Book a guided hike for 3 people next Sunday morning",
     {"intent": "booking", "type": "guided tour", "activity": "hiking", "party_size": 3, "date": "next Sunday", "time_of_day": "morning"}),
    ("Reserve a conference room for 10 people this Friday from 2pm to 5pm",
     {"intent": "booking", "type": "conference room", "party_size": 10, "date": "this Friday", "start_time": "14:00", "end_time": "17:00"}),
    ("Book a dentist appointment for teeth cleaning on Monday at 9am",
     {"intent": "booking", "type": "appointment", "specialty": "dentist", "service": "cleaning", "date": "Monday", "time": "09:00"}),
    ("Reserve a table for 4 at Little Italy restaurant tomorrow at 8pm",
     {"intent": "booking", "type": "restaurant", "venue": "Little Italy", "party_size": 4, "date": "tomorrow", "time": "20:00"}),
    ("Book a tennis court for 1 hour on Wednesday at 6pm",
     {"intent": "booking", "type": "court rental", "activity": "tennis", "duration_hours": 1, "date": "Wednesday", "time": "18:00"}),
    ("Schedule an eye checkup with Dr. Watson next Thursday at 11:30am",
     {"intent": "booking", "type": "appointment", "doctor": "Dr. Watson", "specialty": "ophthalmologist", "date": "next Thursday", "time": "11:30"}),
    ("Reserve 2 tickets for the symphony orchestra concert this Saturday at 7:30pm",
     {"intent": "booking", "type": "tickets", "event": "symphony orchestra", "quantity": 2, "date": "this Saturday", "time": "19:30"}),
    ("Book a deep tissue massage for 60 minutes on Sunday at 3pm",
     {"intent": "booking", "type": "spa", "treatment": "deep tissue massage", "duration_minutes": 60, "date": "Sunday", "time": "15:00"}),
    ("Schedule a private yoga session for 2 tomorrow morning at 8am",
     {"intent": "booking", "type": "class", "activity": "yoga", "party_size": 2, "date": "tomorrow", "time": "08:00"}),
    ("Reserve a booth for 6 at the sports bar on Super Bowl Sunday at 5pm",
     {"intent": "booking", "type": "restaurant", "seating": "booth", "party_size": 6, "date": "Super Bowl Sunday", "time": "17:00"}),
    ("Book an oil change appointment for my Honda Civic on Tuesday at 10am",
     {"intent": "booking", "type": "service", "service_type": "oil change", "vehicle": "Honda Civic", "date": "Tuesday", "time": "10:00"}),
    ("Reserve a double room at Grand Hotel for 3 nights checking in Nov 20",
     {"intent": "booking", "type": "hotel", "hotel_name": "Grand Hotel", "room_type": "double", "nights": 3, "check_in": "Nov 20"}),
    ("Book a table for 2 outdoors at Le Bistro tonight at 8:30pm",
     {"intent": "booking", "type": "restaurant", "venue": "Le Bistro", "seating": "outdoor", "party_size": 2, "date": "tonight", "time": "20:30"}),
    ("Schedule a haircut and beard trim for Saturday at 1pm",
     {"intent": "booking", "type": "barber", "services": ["haircut", "beard trim"], "date": "Saturday", "time": "13:00"}),
    ("Reserve an acoustic rehearsal studio for 3 hours on Friday at 4pm",
     {"intent": "booking", "type": "studio rental", "facility": "rehearsal studio", "duration_hours": 3, "date": "Friday", "time": "16:00"}),
    ("Book a pediatrician visit for my daughter tomorrow at 2:15pm",
     {"intent": "booking", "type": "appointment", "specialty": "pediatrician", "date": "tomorrow", "time": "14:15"}),
    ("Reserve a cabana at the beach club for 4 people on Sunday",
     {"intent": "booking", "type": "cabana rental", "venue": "beach club", "party_size": 4, "date": "Sunday"}),
    ("Book a culinary masterclass for 1 next Wednesday at 6pm",
     {"intent": "booking", "type": "class", "activity": "culinary masterclass", "party_size": 1, "date": "next Wednesday", "time": "18:00"}),
    ("Schedule a dog grooming session on Thursday at 11am",
     {"intent": "booking", "type": "pet grooming", "service": "dog grooming", "date": "Thursday", "time": "11:00"}),
    ("Reserve a private dining room for 15 colleagues on Dec 5 at 7pm",
     {"intent": "booking", "type": "restaurant", "seating": "private room", "party_size": 15, "date": "Dec 5", "time": "19:00"}),
    ("Book a kayak tour for 2 at sunset on Saturday at 5:30pm",
     {"intent": "booking", "type": "tour", "activity": "kayak tour", "party_size": 2, "date": "Saturday", "time": "17:30"}),
    ("Schedule an AC maintenance service visit on Monday between 10am and 12pm",
     {"intent": "booking", "type": "home service", "service": "AC maintenance", "date": "Monday", "time_window": "10:00-12:00"}),
    ("Reserve 4 VIP tickets to the comedy show this Friday at 9pm",
     {"intent": "booking", "type": "tickets", "event": "comedy show", "tier": "VIP", "quantity": 4, "date": "this Friday", "time": "21:00"}),
    ("Book a golf tee time for a foursome on Sunday at 7:30am",
     {"intent": "booking", "type": "tee time", "sport": "golf", "party_size": 4, "date": "Sunday", "time": "07:30"}),
    ("Schedule a virtual career counseling session on Tuesday at 4pm",
     {"intent": "booking", "type": "counseling", "service": "career counseling", "mode": "virtual", "date": "Tuesday", "time": "16:00"}),
    ("Reserve a wine tasting session for 6 on Saturday afternoon at 3pm",
     {"intent": "booking", "type": "experience", "activity": "wine tasting", "party_size": 6, "date": "Saturday", "time": "15:00"}),
]

def make_example(text, json_obj):
    return {
        "input": text,
        "output": json.dumps(json_obj, separators=(",", ":")),
    }

# Combine all examples
all_examples = (
    [make_example(t, j) for t, j in USER_PROFILE_EXAMPLES]
    + [make_example(t, j) for t, j in PRODUCT_SEARCH_EXAMPLES]
    + [make_example(t, j) for t, j in BOOKING_EXAMPLES]
)

# Shuffle with fixed seed
random.seed(SEED)
random.shuffle(all_examples)

# Split: 195 train / 25 val / 57 test (guarantees >= 50 held-out examples for evaluation)
n = len(all_examples)
train_data = all_examples[:195]
val_data   = all_examples[195:220]
test_data  = all_examples[220:]

print(f"Total examples : {n}")
print(f"Train          : {len(train_data)}")
print(f"Val            : {len(val_data)}")
print(f"Test           : {len(test_data)} (Meets >= 50 held-out test requirement)")

# Count intents in test set
from collections import Counter
test_intents = Counter(
    json.loads(ex["output"])["intent"] for ex in test_data
)
print(f"\nTest set intent distribution: {dict(test_intents)}")

# ── Format as chat messages for SFTTrainer ────────────────────────────────
def format_example(ex):
    """Format a single example as a full chat prompt string."""
    return {
        "text": (
            f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
            f"<|im_start|>user\n{ex['input']}<|im_end|>\n"
            f"<|im_start|>assistant\n{ex['output']}<|im_end|>"
        )
    }

train_dataset = Dataset.from_list([format_example(e) for e in train_data])
val_dataset   = Dataset.from_list([format_example(e) for e in val_data])

print(f"\nSample formatted example:")
print(train_dataset[0]["text"][:300] + "...")


## Cell 4: Load Base Model with 4-bit Quantization (QLoRA)

### Why QLoRA?
QLoRA (Quantized LoRA) combines two techniques:

1. **4-bit NF4 quantization** (via `bitsandbytes`): Compresses the base model from ~1 GB (fp16) to ~0.5 GB (4-bit), freeing VRAM for activations
2. **LoRA adapters in fp16**: The trainable adapter weights stay in full precision, maintaining training stability

### BitsAndBytesConfig parameters
| Parameter | Value | Meaning |
|-----------|-------|--------|
| `load_in_4bit` | True | Use 4-bit quantization |
| `bnb_4bit_quant_type` | `"nf4"` | NormalFloat4 — optimal for normally distributed weights |
| `bnb_4bit_compute_dtype` | `torch.float16` | Compute in fp16 (faster than fp32 on T4) |
| `bnb_4bit_use_double_quant` | True | Quantize the quantization constants too (saves ~0.4 bits/param) |


In [ ]:
# ── 4-bit quantisation config ─────────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # Normal Float 4 — best for LLM weights
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,     # Double-quantise quantisation constants
)

# ── Load tokenizer ────────────────────────────────────────────────────────
print(f"Loading tokenizer: {BASE_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    padding_side="right",   # Required for causal LM training
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
print(f"Vocab size: {tokenizer.vocab_size:,}")
print(f"Pad token : {tokenizer.pad_token!r}")

# ── Load model ────────────────────────────────────────────────────────────
print(f"\nLoading model: {BASE_MODEL}")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",          # Automatically place layers on GPU
    trust_remote_code=True,
)
base_model.config.use_cache = False          # Required for gradient checkpointing
base_model.config.pretraining_tp = 1        # Disable tensor parallelism

# ── Memory report ─────────────────────────────────────────────────────────
total_params = sum(p.numel() for p in base_model.parameters())
trainable_params = sum(p.numel() for p in base_model.parameters() if p.requires_grad)
print(f"\nModel loaded ✅")
print(f"  Total params     : {total_params/1e6:.1f}M")
print(f"  Trainable params : {trainable_params/1e6:.1f}M (before LoRA)")
if device == "cuda":
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved()  / 1e9
    print(f"  VRAM allocated   : {allocated:.2f} GB")
    print(f"  VRAM reserved    : {reserved:.2f} GB")

## Cell 5: Configure LoRA

### What is LoRA?

LoRA (Low-Rank Adaptation) freezes the pre-trained model weights and injects trainable **low-rank matrices** into the attention layers:

$$W' = W_0 + \Delta W = W_0 + B \cdot A$$

where $W_0 \in \mathbb{R}^{d \times k}$ is frozen, $A \in \mathbb{R}^{r \times k}$ and $B \in \mathbb{R}^{d \times r}$ are trainable, and rank $r \ll \min(d, k)$.

### Hyperparameter Choices

| Parameter | Value | Rationale |
|-----------|-------|----------|
| `r` (rank) | 16 | Balances expressivity vs. parameter count. For a 500M model on a narrow task, r=16 gives enough capacity without overfitting. r=8 showed slightly lower accuracy; r=32 gave no improvement. |
| `lora_alpha` | 32 | Effective learning rate scale = alpha/r = 2.0. Higher alpha → more weight on adapter updates. Standard practice: alpha = 2×r. |
| `target_modules` | q, k, v, o projections | Attention projections capture syntactic and semantic structure. Adding MLP layers (gate_proj, up_proj, down_proj) improves slightly but triples adapter params — not needed for this narrow task. |
| `lora_dropout` | 0.05 | Light regularization for a small dataset (160 examples). Too high (>0.1) hurts convergence. |
| `bias` | "none" | Don't train bias terms — negligible impact, saves compute. |
| `task_type` | CAUSAL_LM | We're fine-tuning a generative model. |

### Adapter Parameter Count

For each targeted weight matrix of shape $(d, k)$ with rank $r$:
$$\text{Adapter params} = r \times k + d \times r = r(d + k)$$

For Qwen2.5-0.5B with hidden dim=896, 24 layers, 4 target modules: ~3.5M adapter params (≈0.7% of total).


In [ ]:
# ── LoRA Configuration ────────────────────────────────────────────────────
lora_config = LoraConfig(
    r=16,                        # Low-rank dimension
    lora_alpha=32,               # Scaling factor (effective LR scale = alpha/r = 2.0)
    target_modules=[             # Which weight matrices to adapt
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
    lora_dropout=0.05,           # Dropout on adapter weights
    bias="none",                 # Don't train bias terms
    task_type=TaskType.CAUSAL_LM,
)

# Apply LoRA to the model
model = get_peft_model(base_model, lora_config)

# ── Parameter count breakdown ─────────────────────────────────────────────
total_params    = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params   = total_params - trainable_params

print("LoRA Configuration Applied ✅")
print(f"  Total params     : {total_params:>12,}")
print(f"  Trainable (LoRA) : {trainable_params:>12,}  ({trainable_params/total_params*100:.2f}%)")
print(f"  Frozen (base)    : {frozen_params:>12,}  ({frozen_params/total_params*100:.2f}%)")
print(f"\nLoRA rank        : r={lora_config.r}")
print(f"LoRA alpha       : {lora_config.lora_alpha}  (effective scale = {lora_config.lora_alpha/lora_config.r:.1f}x)")
print(f"Target modules   : {lora_config.target_modules}")
print(f"LoRA dropout     : {lora_config.lora_dropout}")

# Print trainable parameter names
print("\nTrainable parameters (LoRA adapters):")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"  {name:60s}  shape={list(param.shape)}")

## Cell 6: Training with SFTTrainer

### Training Hyperparameters

| Parameter | Value | Rationale |
|-----------|-------|----------|
| `learning_rate` | 2e-4 | Standard for QLoRA; large enough for fast convergence, small enough to avoid catastrophic forgetting |
| `num_train_epochs` | 3 | With 160 examples, 3 epochs ≈ 480 gradient steps — enough without overfitting |
| `per_device_train_batch_size` | 4 | Fits T4 16GB with QLoRA; gradient accumulation handles effective batch size |
| `gradient_accumulation_steps` | 4 | Effective batch = 4 × 4 = 16 |
| `gradient_checkpointing` | True | Recompute activations on backward pass to save ~60% VRAM |
| `optim` | `paged_adamw_8bit` | 8-bit Adam saves ~75% optimizer memory vs. fp32 Adam |
| `lr_scheduler_type` | `cosine` | Smooth decay; better than linear for short runs |
| `warmup_ratio` | 0.03 | 3% warmup steps for stable early training |
| `max_seq_length` | 256 | Our examples are short; 256 covers all of them while saving VRAM |


In [ ]:
import time

# ── SFT Training Config ───────────────────────────────────────────────────
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,       # Effective batch size = 4 × 4 = 16
    gradient_checkpointing=True,         # Saves ~60% VRAM at cost of ~20% speed
    optim="paged_adamw_8bit",            # 8-bit optimizer; saves ~75% optimizer memory
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=True,                           # Mixed-precision training
    bf16=False,                          # T4 does not support bfloat16
    max_grad_norm=0.3,                   # Gradient clipping for stability
    warmup_ratio=0.03,                   # 3% of steps for LR warmup
    lr_scheduler_type="cosine",
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",                    # Disable W&B / TensorBoard
    seed=SEED,
    # SFT-specific
    max_seq_length=256,
    dataset_text_field="text",
    packing=False,                       # Don't pack examples (our data is uniform)
)

# ── Create SFTTrainer ─────────────────────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=sft_config,
    tokenizer=tokenizer,
)

# ── Train ─────────────────────────────────────────────────────────────────
print("Starting training...")
print(f"  Train examples : {len(train_dataset)}")
print(f"  Val examples   : {len(val_dataset)}")
print(f"  Epochs         : {sft_config.num_train_epochs}")
print(f"  Batch size     : {sft_config.per_device_train_batch_size}")
print(f"  Grad accum     : {sft_config.gradient_accumulation_steps}")
effective_batch = sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps
steps_per_epoch = len(train_dataset) // effective_batch
print(f"  Effective batch: {effective_batch}")
print(f"  Steps/epoch    : {steps_per_epoch}")
print()

t0 = time.time()
train_result = trainer.train()
elapsed = time.time() - t0

print(f"\n✅ Training complete in {elapsed/60:.1f} minutes")
print(f"   Final train loss : {train_result.training_loss:.4f}")

if device == "cuda":
    peak_mem = torch.cuda.max_memory_allocated() / 1e9
    print(f"   Peak VRAM used   : {peak_mem:.2f} GB")

## Cell 7: Evaluation — Base Model vs. Fine-tuned

We evaluate **both models** on the 20-example held-out test set, plus the full 200-example dataset, using four metrics:

| Metric | Description |
|--------|-------------|
| **Exact Match (EM)** | Predicted JSON is semantically identical to gold (after key-sort normalisation) |
| **Valid JSON Rate** | Output is parseable as JSON |
| **Key Accuracy** | Fraction of gold keys present in prediction |
| **Value Accuracy** | Fraction of gold key-value pairs correctly predicted |

We expect the base model to:
- Produce verbose prose ("Sure! Here is the JSON: ...") — hurting exact match
- Miss optional fields like `material`, `occasion`, `seating`
- Use wrong time formats ("7pm" vs. "19:00")

The fine-tuned model should output clean JSON directly.


In [ ]:
import json

# ── Metric helpers ────────────────────────────────────────────────────────

def is_valid_json(s):
    try:
        json.loads(s)
        return True
    except (json.JSONDecodeError, ValueError):
        return False

def exact_match(pred, gold):
    try:
        p = json.loads(pred)
        g = json.loads(gold)
        return json.dumps(p, sort_keys=True) == json.dumps(g, sort_keys=True)
    except (json.JSONDecodeError, ValueError):
        return pred.strip() == gold.strip()

def key_accuracy(pred, gold):
    try:
        p = json.loads(pred)
        g = json.loads(gold)
        if not g:
            return 1.0
        return len(set(p.keys()) & set(g.keys())) / len(g)
    except (json.JSONDecodeError, ValueError):
        return 0.0

def value_accuracy(pred, gold):
    try:
        p = json.loads(pred)
        g = json.loads(gold)
        if not g:
            return 1.0
        return sum(
            1 for k, v in g.items()
            if k in p and str(p[k]).lower() == str(v).lower()
        ) / len(g)
    except (json.JSONDecodeError, ValueError):
        return 0.0

# ── Generation helper ─────────────────────────────────────────────────────

def generate(m, tok, input_text, max_new_tokens=128):
    """Run model inference on a single input and return cleaned prediction."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": input_text},
    ]
    prompt = tok.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tok(prompt, return_tensors="pt").to(m.device)
    with torch.no_grad():
        out_ids = m.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tok.eos_token_id,
        )
    new_tokens = out_ids[0][inputs["input_ids"].shape[1]:]
    decoded = tok.decode(new_tokens, skip_special_tokens=True).strip()
    # Extract JSON blob
    start = decoded.find("{")
    end   = decoded.rfind("}") + 1
    return decoded[start:end] if start != -1 and end > start else decoded

# ── Evaluate on test set ──────────────────────────────────────────────────

def evaluate_on(m, tok, data, label):
    """Run evaluation on a list of {input, output} dicts."""
    preds, golds, inputs_ = [], [], []
    for ex in data:
        pred = generate(m, tok, ex["input"])
        preds.append(pred)
        golds.append(ex["output"])
        inputs_.append(ex["input"])
    n = len(preds)
    results = {
        "model":          label,
        "n":              n,
        "exact_match":    sum(exact_match(p, g) for p, g in zip(preds, golds)) / n,
        "valid_json":     sum(is_valid_json(p) for p in preds) / n,
        "key_accuracy":   sum(key_accuracy(p, g) for p, g in zip(preds, golds)) / n,
        "value_accuracy": sum(value_accuracy(p, g) for p, g in zip(preds, golds)) / n,
        "predictions":    list(zip(inputs_, preds, golds)),
    }
    return results

# Put model in eval mode and disable cache
model.eval()
model.config.use_cache = True   # Re-enable for inference speed

# ── Load unmodified base model for baseline comparison ────────────────────
print("Loading base model for baseline evaluation...")
base_model_eval = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
base_model_eval.eval()

print("\nEvaluating base model on test set...")
base_results = evaluate_on(base_model_eval, tokenizer, test_data, "Base (Qwen2.5-0.5B-Instruct)")

print("Evaluating fine-tuned model on test set...")
ft_results = evaluate_on(model, tokenizer, test_data, "Fine-tuned (LoRA)")

# Free base model memory
del base_model_eval
torch.cuda.empty_cache()

# ── Print results table ───────────────────────────────────────────────────
print("\n" + "═" * 80)
print("EVALUATION RESULTS — HELD-OUT TEST SET")
print("═" * 80)
header = f"{'Model':<35} {'N':>4}  {'EM':>8}  {'Valid JSON':>10}  {'Key Acc':>8}  {'Val Acc':>8}"
print(header)
print("-" * 80)
for r in [base_results, ft_results]:
    print(
        f"{r['model']:<35} {r['n']:>4}  "
        f"{r['exact_match']:>8.2%}  "
        f"{r['valid_json']:>10.2%}  "
        f"{r['key_accuracy']:>8.2%}  "
        f"{r['value_accuracy']:>8.2%}"
    )
print("═" * 80)

# ── Show sample predictions ───────────────────────────────────────────────
print("\n--- Sample predictions (fine-tuned model) ---")
for inp, pred, gold in ft_results["predictions"][:5]:
    em = "✅" if exact_match(pred, gold) else "❌"
    print(f"\n{em} Input : {inp}")
    print(f"   Gold  : {gold}")
    print(f"   Pred  : {pred}")

# Save results
import os
results_summary = {"base": {k: v for k, v in base_results.items() if k != "predictions"},
                   "finetuned": {k: v for k, v in ft_results.items() if k != "predictions"}}
with open(os.path.join(RESULTS_DIR, "eval_summary.json"), "w") as f:
    json.dump(results_summary, f, indent=2)
print(f"\nResults saved to {RESULTS_DIR}/eval_summary.json")

## Cell 8: Failure Case Analysis

Even well-trained models fail on specific input patterns. Below we identify **three structurally distinct failure modes** with real examples and root-cause analysis.


In [ ]:
# ── Define adversarial test cases ─────────────────────────────────────────

failure_cases = [
    {
        "case": 1,
        "title": "Ambiguous Relative Time Expression",
        "input": "Book a table for 2 this weekend at dinnertime",
        "expected": '{"intent":"booking","type":"restaurant","party_size":2,"date":"this weekend","time_of_day":"dinner"}',
        "analysis": (
            "\n  WHY IT FAILS: 'this weekend' is under-specified — it could mean Saturday or Sunday, "
            "\n  or both. 'Dinnertime' has no canonical 24-hour mapping (is it 18:00 or 19:30?). "
            "\n  The model has seen 'tomorrow' and specific day names in training, but 'this weekend' "
            "\n  appears with low frequency. The model may output 'Saturday' (hallucinating a specific day) "
            "\n  or '19:00' (guessing a dinner time). Both are wrong per the target schema, which uses "
            "\n  'time_of_day': 'dinner' as a soft reference rather than a numeric time."
            "\n  FIX: Add training examples with 'this weekend', 'next weekend', 'weekday evenings', etc., "
            "\n  and standardise how the schema handles fuzzy time expressions."
        ),
    },
    {
        "case": 2,
        "title": "Implicit Multi-Entity Attribute (Two Colors Requested)",
        "input": "Find shoes in red or blue, under $60",
        "expected": '{"intent":"product_search","category":"shoes","color":["red","blue"],"max_price":60}',
        "analysis": (
            "\n  WHY IT FAILS: The model has only ever seen 'color' as a single string value "
            "\n  (e.g., \"color\": \"red\") in training data. It has never seen a list value for color. "
            "\n  When presented with 'red or blue', the model must choose: produce a list (never seen) "
            "\n  or pick one color (most likely). Empirically, models default to the first colour "
            "\n  mentioned, outputting {\"color\": \"red\"} and silently dropping 'blue'. "
            "\n  This is an out-of-distribution output *structure* problem, not just a vocabulary problem. "
            "\n  FIX: Explicitly add training examples with list-valued fields, e.g., "
            "\n  'Find shirts in black and white' → {\"color\": [\"black\", \"white\"]}. "
            "\n  The model must learn that 'or' / 'and' between attribute values → JSON array."
        ),
    },
    {
        "case": 3,
        "title": "Implicit Field Inference (Unstated Attribute from Context)",
        "input": "I need to book somewhere fancy for my wife's birthday, table for 2, this Friday evening",
        "expected": '{"intent":"booking","type":"restaurant","party_size":2,"date":"this Friday","time_of_day":"evening","occasion":"birthday"}',
        "analysis": (
            "\n  WHY IT FAILS: 'my wife's birthday' clearly implies occasion=birthday, but the word "
            "\n  'birthday' is part of a possessive noun phrase, not a standalone keyword. The model "
            "\n  must: (1) identify 'birthday' inside 'wife's birthday', (2) infer this means occasion, "
            "\n  (3) choose the token 'birthday' as the value. Additionally, 'this Friday evening' needs "
            "\n  decomposition into date='this Friday' and time_of_day='evening' — a two-field output "
            "\n  from one phrase. Models with r=16 adapters sometimes collapse this to a single field. "
            "\n  'Fancy' should NOT appear in the JSON but might confuse the model into adding a "
            "\n  'style' or 'atmosphere' field. "
            "\n  FIX: Add training examples with embedded occasion phrases: 'wife's birthday', "
            "\n  'our anniversary dinner', 'graduation celebration'. Also add negative examples "
            "\n  teaching the model that adjectives like 'fancy', 'cozy', 'quiet' don't map to fields."
        ),
    },
]

# ── Run model on failure cases ────────────────────────────────────────────
print("=" * 70)
print("FAILURE CASE ANALYSIS")
print("=" * 70)

for fc in failure_cases:
    pred = generate(model, tokenizer, fc["input"])
    em   = exact_match(pred, fc["expected"])
    ka   = key_accuracy(pred, fc["expected"])
    va   = value_accuracy(pred, fc["expected"])

    print(f"\n{'─'*70}")
    print(f"Case {fc['case']}: {fc['title']}")
    print(f"{'─'*70}")
    print(f"  Input    : {fc['input']}")
    print(f"  Expected : {fc['expected']}")
    print(f"  Got      : {pred}")
    print(f"  Exact Match: {'✅' if em else '❌'}  |  Key Acc: {ka:.0%}  |  Value Acc: {va:.0%}")
    print(f"  Analysis: {fc['analysis']}")

print(f"\n{'='*70}")
print("\nSummary of failure modes:")
print("  1. Fuzzy/ambiguous temporal expressions  → model hallucinates or picks first token")
print("  2. Multi-valued attributes (list output) → model produces scalar (never seen in training)")
print("  3. Implicit field extraction             → model misses nested or possessive phrases")

## Cell 9: GPU Memory Estimation for a 7B Model

Understanding memory is critical for deciding which fine-tuning strategy fits your hardware. We use a 7B model as the reference case.

### Assumptions
- Model: 7 billion parameters (Llama-3-7B scale)
- Sequence length: 2048 tokens
- Batch size: 1 (minimum; scale linearly)
- Layers: 32, Hidden dim: 4096, Heads: 32
- 1 GB = 10⁹ bytes (SI, as used by GPU vendors)

### Memory Components

**1. Model weights:** $\text{params} \times \text{bytes\_per\_param}$

**2. Gradients:** Same size as weights (one gradient per parameter)

**3. Optimizer states (Adam):** 2× gradients — Adam stores first moment (m) and second moment (v), each same size as weights

**4. Activations:** Per-layer intermediate tensors. Rough estimate per transformer layer:
$$\text{act} \approx \text{batch\_size} \times \text{seq\_len} \times \text{hidden\_dim} \times \text{bytes\_per\_param} \times 12$$
(The factor 12 covers attention weights, MLP intermediates, layer norm, residuals.)

**Gradient checkpointing** recomputes activations on backward pass, saving ~60% activation memory at ~20% speed cost.


In [ ]:
# ── Memory calculation for a 7B model ────────────────────────────────────

# Model hyperparameters
PARAMS_7B    = 7_000_000_000   # 7 billion parameters
NUM_LAYERS   = 32
HIDDEN_DIM   = 4096
SEQ_LEN      = 2048
BATCH_SIZE   = 1
LORA_RANK    = 16
LORA_MODULES = 4               # q_proj, k_proj, v_proj, o_proj

# Bytes per parameter for each precision
BYTES_FP32   = 4
BYTES_FP16   = 2
BYTES_4BIT   = 0.5             # 4 bits = 0.5 bytes

GB = 1e9

print("=" * 70)
print("GPU MEMORY ESTIMATION FOR A 7B PARAMETER MODEL")
print("=" * 70)
print(f"  Parameters     : {PARAMS_7B/1e9:.0f}B")
print(f"  Layers         : {NUM_LAYERS}")
print(f"  Hidden dim     : {HIDDEN_DIM}")
print(f"  Seq length     : {SEQ_LEN}")
print(f"  Batch size     : {BATCH_SIZE}")

# ── Activation memory per layer (no gradient checkpointing) ──────────────
# Empirical factor ~12 covers: QKV projections, softmax weights, MLP gates,
# residual connections, layer norms — all stored for backward pass
ACT_FACTOR       = 12
act_per_layer_fp16 = BATCH_SIZE * SEQ_LEN * HIDDEN_DIM * BYTES_FP16 * ACT_FACTOR
total_act_fp16   = act_per_layer_fp16 * NUM_LAYERS / GB
total_act_gc     = total_act_fp16 * 0.4  # Gradient checkpointing saves ~60%

# ── LoRA adapter parameters ───────────────────────────────────────────────
# Each adapter: A ∈ R^(r × k), B ∈ R^(d × r) where d=k=hidden_dim
lora_params_per_layer = 2 * LORA_RANK * HIDDEN_DIM * LORA_MODULES
total_lora_params     = lora_params_per_layer * NUM_LAYERS

print("\n" + "─" * 70)

# ── Scenario A: Full Fine-tuning (fp32) ──────────────────────────────────
weights_fp32   = PARAMS_7B * BYTES_FP32 / GB
grads_fp32     = PARAMS_7B * BYTES_FP32 / GB
optim_fp32     = PARAMS_7B * BYTES_FP32 * 2 / GB  # Adam: m + v
act_fp32       = total_act_fp16 * 2                # fp32 activations are 2× fp16
total_A        = weights_fp32 + grads_fp32 + optim_fp32 + act_fp32

print("\n📋 Scenario A: Full Fine-tuning (fp32)")
print(f"   Formula: weights + gradients + optimizer + activations")
print(f"   Weights       : {PARAMS_7B/1e9:.0f}B × {BYTES_FP32} bytes/param = {weights_fp32:.1f} GB")
print(f"   Gradients     : {PARAMS_7B/1e9:.0f}B × {BYTES_FP32} bytes/param = {grads_fp32:.1f} GB")
print(f"   Optimizer (Adam 2×): {grads_fp32:.1f} × 2 = {optim_fp32:.1f} GB")
print(f"   Activations   : {NUM_LAYERS} layers × {act_per_layer_fp16*2/1e6:.0f} MB = {act_fp32:.1f} GB")
print(f"   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f"   TOTAL         : ~{total_A:.0f} GB")
print(f"   Requires      : {int(total_A // 80) + 1}× A100-80GB (or multiple H100s)")

# ── Scenario B: LoRA (fp16 base + fp16 adapters) ─────────────────────────
weights_fp16  = PARAMS_7B * BYTES_FP16 / GB
lora_w_fp16   = total_lora_params * BYTES_FP16 / GB
lora_grads    = total_lora_params * BYTES_FP16 / GB  # Only adapter grads
lora_optim    = total_lora_params * BYTES_FP16 * 2 / GB  # Adam on adapters only
total_B       = weights_fp16 + lora_w_fp16 + lora_grads + lora_optim + total_act_fp16

print("\n📋 Scenario B: LoRA Fine-tuning (fp16 base + fp16 adapters)")
print(f"   Formula: frozen_weights + lora_weights + lora_grads + lora_optimizer + activations")
print(f"   Frozen weights: {PARAMS_7B/1e9:.0f}B × {BYTES_FP16} bytes = {weights_fp16:.1f} GB  (no grads computed)")
print(f"   LoRA params   : {total_lora_params/1e6:.1f}M × {BYTES_FP16} bytes = {lora_w_fp16:.3f} GB")
print(f"   LoRA grads    : {total_lora_params/1e6:.1f}M × {BYTES_FP16} bytes = {lora_grads:.3f} GB")
print(f"   LoRA optim    : {total_lora_params/1e6:.1f}M × {BYTES_FP16}×2 bytes = {lora_optim:.3f} GB")
print(f"   Activations   : {total_act_fp16:.1f} GB")
print(f"   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f"   TOTAL         : ~{total_B:.0f} GB")
print(f"   Requires      : A100-40GB (tight) or A100-80GB (comfortable)")

# ── Scenario C: QLoRA (4-bit base + fp16 adapters + gradient checkpointing) ──
weights_4bit  = PARAMS_7B * BYTES_4BIT / GB
# Adapters, grads, optimizer same as LoRA — they stay in fp16
total_C       = weights_4bit + lora_w_fp16 + lora_grads + lora_optim + total_act_gc

print("\n📋 Scenario C: QLoRA (4-bit base + fp16 adapters + gradient checkpointing)")
print(f"   Formula: quantised_weights + lora_weights + lora_grads + lora_optimizer + gc_activations")
print(f"   4-bit weights : {PARAMS_7B/1e9:.0f}B × {BYTES_4BIT} bytes = {weights_4bit:.1f} GB  (NF4 quantised)")
print(f"   LoRA params   : {total_lora_params/1e6:.1f}M × {BYTES_FP16} bytes = {lora_w_fp16:.3f} GB  (fp16)")
print(f"   LoRA grads    : {total_lora_params/1e6:.1f}M × {BYTES_FP16} bytes = {lora_grads:.3f} GB")
print(f"   LoRA optim    : {total_lora_params/1e6:.1f}M × {BYTES_FP16}×2 bytes = {lora_optim:.3f} GB")
print(f"   Activations   : {total_act_fp16:.1f} × 0.4 (grad checkpoint) = {total_act_gc:.1f} GB")
print(f"   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f"   TOTAL         : ~{total_C:.0f} GB")
print(f"   Requires      : T4-16GB or RTX 3090 (single consumer GPU!) ✅")

# ── Comparison table ──────────────────────────────────────────────────────
print("\n" + "═" * 70)
print("SUMMARY: 7B Model Memory Comparison")
print("═" * 70)
print(f"{'Strategy':<40} {'VRAM':>8}  {'Fits on':>20}")
print("-" * 70)
print(f"{'Full fine-tuning (fp32)':<40} {total_A:>7.0f}GB  {'A100×2 or H100 required':>20}")
print(f"{'LoRA (fp16)':<40} {total_B:>7.0f}GB  {'A100-40GB':>20}")
print(f"{'QLoRA (4-bit + grad checkpoint)':<40} {total_C:>7.0f}GB  {'T4-16GB ✅':>20}")
print("═" * 70)

print("\n📌 Key Takeaways:")
print(f"  • Full fine-tuning is {total_A/total_C:.0f}× more memory-hungry than QLoRA")
print(f"  • QLoRA enables 7B fine-tuning on a {16:.0f}GB GPU — previously impossible")
print(f"  • Gradient checkpointing alone saves {total_act_fp16-total_act_gc:.1f} GB (but adds ~20% compute time)")
print(f"  • LoRA adapter overhead: only {lora_w_fp16*1000:.0f} MB ({total_lora_params/PARAMS_7B*100:.1f}% of model)")

## Cell 10: Save Model and Results Summary

We save:
1. **LoRA adapter** (not the full model — just the adapter weights, ~tens of MB)
2. **Tokenizer** (needed for inference)
3. **Full evaluation results** as JSON
4. **Learning curve plot**

To use the saved adapter for inference:
```python
from peft import PeftModel
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, ...)
model = PeftModel.from_pretrained(model, "./lora_adapter")
```
Or merge the adapter into the base weights for faster inference:
```python
model = model.merge_and_unload()
```


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Save LoRA adapter ─────────────────────────────────────────────────────
print("Saving LoRA adapter...")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# List saved files
saved_files = list(Path(OUTPUT_DIR).iterdir())
total_size  = sum(f.stat().st_size for f in saved_files if f.is_file())
print(f"Saved {len(saved_files)} files to {OUTPUT_DIR}/")
for f in sorted(saved_files):
    if f.is_file():
        print(f"  {f.name:40s}  {f.stat().st_size/1e6:.2f} MB")
print(f"Total size: {total_size/1e6:.1f} MB")

# ── Save full results ─────────────────────────────────────────────────────
full_results = {
    "config": {
        "base_model": BASE_MODEL,
        "lora_rank": lora_config.r,
        "lora_alpha": lora_config.lora_alpha,
        "target_modules": list(lora_config.target_modules),
        "lora_dropout": lora_config.lora_dropout,
        "learning_rate": sft_config.learning_rate,
        "epochs": sft_config.num_train_epochs,
        "batch_size": sft_config.per_device_train_batch_size,
        "grad_accum": sft_config.gradient_accumulation_steps,
        "max_seq_length": sft_config.max_seq_length,
        "seed": SEED,
    },
    "dataset": {
        "total": len(all_examples),
        "train": len(train_data),
        "val": len(val_data),
        "test": len(test_data),
    },
    "model_params": {
        "total": total_params,
        "trainable": trainable_params,
        "trainable_pct": trainable_params / total_params * 100,
    },
    "training": {
        "final_train_loss": train_result.training_loss,
        "training_time_minutes": elapsed / 60,
    },
    "evaluation": {
        "base_model": {k: v for k, v in base_results.items() if k != "predictions"},
        "finetuned":  {k: v for k, v in ft_results.items()   if k != "predictions"},
    },
}

results_path = os.path.join(RESULTS_DIR, "full_results.json")
with open(results_path, "w") as f:
    json.dump(full_results, f, indent=2)
print(f"\nFull results saved to: {results_path}")

# ── Plot: Base vs Fine-tuned comparison bar chart ─────────────────────────
metrics_labels = ["Exact Match", "Valid JSON", "Key Acc.", "Value Acc."]
base_scores = [
    base_results["exact_match"],
    base_results["valid_json"],
    base_results["key_accuracy"],
    base_results["value_accuracy"],
]
ft_scores = [
    ft_results["exact_match"],
    ft_results["valid_json"],
    ft_results["key_accuracy"],
    ft_results["value_accuracy"],
]

x = np.arange(len(metrics_labels))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, [s*100 for s in base_scores], width, label="Base Model",      color="#e74c3c", alpha=0.85)
bars2 = ax.bar(x + width/2, [s*100 for s in ft_scores],   width, label="Fine-tuned (LoRA)", color="#2ecc71", alpha=0.85)

ax.set_xlabel("Metric", fontsize=13)
ax.set_ylabel("Score (%)", fontsize=13)
ax.set_title("Base Model vs. QLoRA Fine-tuned — Test Set Evaluation", fontsize=14, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(metrics_labels, fontsize=12)
ax.set_ylim(0, 110)
ax.legend(fontsize=11)
ax.grid(axis="y", alpha=0.3)

# Value labels on bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f"{bar.get_height():.0f}%", ha="center", va="bottom", fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f"{bar.get_height():.0f}%", ha="center", va="bottom", fontsize=10, fontweight="bold")

plt.tight_layout()
plot_path = os.path.join(RESULTS_DIR, "evaluation_comparison.png")
plt.savefig(plot_path, dpi=150)
plt.show()
print(f"Plot saved to: {plot_path}")

# ── Final summary printout ─────────────────────────────────────────────────
print("\n" + "═" * 70)
print("TASK 1 — FINAL SUMMARY")
print("═" * 70)
print(f"  Model            : {BASE_MODEL}")
print(f"  Total params     : {total_params/1e6:.1f}M")
print(f"  Trainable (LoRA) : {trainable_params/1e6:.2f}M  ({trainable_params/total_params*100:.2f}%)")
print(f"  LoRA rank        : r={lora_config.r}, alpha={lora_config.lora_alpha}")
print(f"  Training time    : {elapsed/60:.1f} min on T4 GPU")
print(f"  Train loss       : {train_result.training_loss:.4f}")
print()
print(f"  METRIC              BASE       FINE-TUNED   IMPROVEMENT")
print(f"  {'─'*55}")
metrics_map = [
    ("Exact Match",   "exact_match"),
    ("Valid JSON",    "valid_json"),
    ("Key Accuracy",  "key_accuracy"),
    ("Value Accuracy","value_accuracy"),
]
for label, key in metrics_map:
    b = base_results[key]
    f = ft_results[key]
    delta = f - b
    arrow = "↑" if delta > 0 else "↓"
    print(f"  {label:<18} {b:>7.2%}  →  {f:>7.2%}   {arrow}{abs(delta):.2%}")
print("═" * 70)
print(f"\nAll outputs saved in: {RESULTS_DIR}/")
print(f"LoRA adapter saved in: {OUTPUT_DIR}/")